# URL to Audio Converter using Gemini 2.5 TTS

This notebook demonstrates how to convert an article from a URL into audio using Google's Gemini 2.5 Text-to-Speech models via the Vertex AI API.

## Overview

1.  **Extract Text**: We'll use `beautifulsoup4` (or a fallback) to extract the main text from a given URL.
2.  **Synthesize Audio**: We'll use the `google-genai` SDK to convert the text to speech using models like `gemini-2.5-pro-preview-tts` or `gemini-2.5-flash-tts`.

## Prerequisites
- Google Cloud Project with Vertex AI API enabled.
- Python environment with necessary libraries.

### Install Dependencies

In [ ]:
%pip install --upgrade --quiet google-genai beautifulsoup4 requests

: 

### Authenticate (Colab)

In [ ]:
import sys
if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

: 

### Configure Project and Client

In [ ]:
import os
from google import genai
from google.genai import types

# Set your project ID here
PROJECT_ID = "customer-demo-01" # @param {type:"string"}
LOCATION = "us-central1" # @param {type:"string"}

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

### Text Extraction Utility

In [ ]:
import requests
import re

def extract_text_from_url(url):
    """
    Extracts text content from a given URL.
    Attempts to use BeautifulSoup if available, otherwise falls back to regex.
    """
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        html_content = response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL: {e}")
        return None

    try:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # Remove script and style elements
        for script in soup(["script", "style"]):
            script.extract()
            
        text = soup.get_text()
        
        # Break into lines and remove leading/trailing space on each
        lines = (line.strip() for line in text.splitlines())
        # Break multi-headlines into a line each
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        # Drop blank lines
        text = '\n'.join(chunk for chunk in chunks if chunk)
        
        return text
    except ImportError:
        print("BeautifulSoup not found. Using simple regex fallback.")
        # Regex fallback
        clean = re.sub(r'<(script|style).*?>.*?</\1>', '', html_content, flags=re.DOTALL)
        clean = re.sub(r'<.*?>', '', clean)
        clean = clean.replace('&nbsp;', ' ').replace('&lt;', '<').replace('&gt;', '>').replace('&amp;', '&')
        clean = re.sub(r'\s+', ' ', clean).strip()
        return clean

### Speech Synthesis Utility

In [ ]:
from IPython.display import Audio, display

def synthesize_and_play(text, model="gemini-2.5-pro-tts", voice="Aoede"):
    print(f"Synthesizing {len(text)} chars with model={model}, voice={voice}...")
    
    # Truncate for demo if huge
    if len(text) > 4000:
        print("Text > 4000 chars, truncating for demo purposes.")
        text = text[:4000]

    try:
        response = client.models.generate_content(
            model=model,
            contents=f"Please read this text out loud naturally: {text}",
            config=types.GenerateContentConfig(
                response_modalities=["AUDIO"],
                speech_config=types.SpeechConfig(
                    voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(
                            voice_name=voice
                        )
                    )
                )
            )
        )

        if response.candidates and response.candidates[0].content.parts:
             for part in response.candidates[0].content.parts:
                if part.inline_data:
                    audio_bytes = part.inline_data.data
                    display(Audio(audio_bytes, autoplay=False))
                    return audio_bytes
        print("No audio returned.")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

### Run URL to Audio

In [ ]:
URL = "https://www.lefigaro.fr/en/in-dubai-where-more-and-more-french-flock-to-start-a-new-life-20260124" # @param {type:"string"}
MODEL_NAME = "gemini-2.5-pro-preview-tts" # @param ["gemini-2.5-pro-preview-tts", "gemini-2.5-flash-tts", "gemini-2.5-pro-tts"]
VOICE_NAME = "Aoede" # @param ["Aoede", "Kore", "Fenrir"]

print(f"Fetching {URL}...")
text_content = extract_text_from_url(URL)

if text_content:
    print(f"Extracted {len(text_content)} characters.")
    print("Preview:", text_content[:200])
    audio_data = synthesize_and_play(text_content, model=MODEL_NAME, voice=VOICE_NAME)
else:
    print("Failed to extract text.")